<a href="https://colab.research.google.com/github/Dolmaa24/tunekit/blob/main/notebooks/tunekit_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# tunekit — QLoRA fine-tuning in Colab

Fine-tunes an open LLM in 4-bit (QLoRA) on Colab's free T4, then evaluates, chats with, and exports it.

**Before you start: Runtime → Change runtime type → T4 GPU.** Without a GPU, 4-bit training cannot run
(bitsandbytes is CUDA-only) and the notebook will tell you so in step 0.

In [2]:
#@title Step 0 — check the GPU and install tunekit
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader || echo "NO GPU: Runtime -> Change runtime type -> T4 GPU, then rerun"
# Colab preinstalls torchao 0.10, which current peft rejects outright (ImportError at
# adapter-load time). tunekit does not use torchao, so removing it is the clean fix.
!pip uninstall -q -y torchao 2>/dev/null
!pip install -q "tunekit[quant] @ git+https://github.com/Dolmaa24/tunekit"
# 'bitsandbytes (QLoRA): available' below is the precondition for 4-bit training.
!tunekit info

Tesla T4, 15360 MiB, 580.82.07
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.5/925.5 kB 58.9 MB/s eta 0:00:00
tunekit 0.1.0
hardware: cuda: Tesla T4, 15.6 GB (bf16=yes)
bitsandbytes (QLoRA): available


## 1. Data

tunekit reads the first row and picks the converter, so these all work as-is:

| format | shape |
|---|---|
| messages | `{"messages": [{"role": "user", "content": "..."}, {"role": "assistant", "content": "..."}]}` |
| alpaca | `{"instruction": "...", "input": "...", "output": "..."}` |
| sharegpt | `{"conversations": [{"from": "human", "value": "..."}, {"from": "gpt", "value": "..."}]}` |
| prompt/completion | `{"prompt": "...", "completion": "..."}` |
| text | `{"text": "..."}` |

The cell below uses a small bundled example. Set `UPLOAD = True` to pick your own file instead, or
set `DATA` to a Hugging Face dataset id (e.g. `"yahma/alpaca-cleaned"`).

In [3]:
UPLOAD = False   #@param {type:"boolean"}

if UPLOAD:
    from google.colab import files
    DATA = next(iter(files.upload()))          # blocks until you choose a file
else:
    !curl -sL -o pirate.jsonl https://raw.githubusercontent.com/Dolmaa24/tunekit/main/examples/pirate.jsonl
    DATA = "pirate.jsonl"
print("using:", DATA)

using: pirate.jsonl


In [4]:
#@title Step 2 — inspect the data before spending GPU time
# Shows the detected format, the exact text the model will train on, and token-length percentiles.
!tunekit validate --model Qwen/Qwen2.5-1.5B-Instruct --data {DATA}

Generating train split: 100 examples [00:00, 10870.30 examples/s]
format: messages -> messages
rows: 98 train / 2 eval / 0 dropped
config.json: 100% 660/660 [00:00<00:00, 3.10MB/s]
tokenizer_config.json: 100% 7.30k/7.30k [00:00<00:00, 26.1MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 67.9MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 127MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 140MB/s]
      token lengths (tokenizer:       
 Qwen/Qwen2.5-1.5B-Instruct, sampled  
                98/98)                
┏━━━━━┳━━━━━┳━━━━━┳━━━━━┳━━━━━┳━━━━━━┓
┃ min ┃ p50 ┃ p90 ┃ p99 ┃ max ┃ mean ┃
┡━━━━━╇━━━━━╇━━━━━╇━━━━━╇━━━━━╇━━━━━━┩
│  44 │  61 │  69 │  73 │  73 │ 60.7 │
└─────┴─────┴─────┴─────┴─────┴──────┘
~5,953 tokens per epoch
────────────────────────────────── example 0 ───────────────────────────────────
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Say hello.<|im_end|>
<|im_start|>assistant
Ahoy there, me hear

## 3. Train in 4-bit (QLoRA)

`model.load_in_4bit=true` loads the base model as 4-bit NF4 and trains LoRA adapters on top. On a free T4:

| model | mode | VRAM |
|---|---|---|
| `Qwen/Qwen2.5-1.5B-Instruct` | QLoRA 4-bit | ~3 GB |
| `Qwen/Qwen2.5-7B-Instruct` | QLoRA 4-bit | ~12 GB |
| `meta-llama/Llama-3.2-3B-Instruct` | QLoRA 4-bit | ~6 GB (needs a HF token + license acceptance) |

`tunekit info --model` reports the weight size against this GPU before anything downloads.

In [5]:
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"   #@param {type:"string"}

!tunekit info --model {MODEL}

!tunekit train --model {MODEL} --data {DATA} --output outputs/qlora \
    --set model.load_in_4bit=true \
    --set train.epochs=2 --set train.batch_size=4 --set train.grad_accum=4 \
    --set train.optimizer=paged_adamw_8bit --set data.max_length=1024

tunekit 0.1.0
hardware: cuda: Tesla T4, 15.6 GB (bf16=yes)
bitsandbytes (QLoRA): available
Qwen/Qwen2.5-1.5B-Instruct: 3.1 GB of weights on disk
  resident base weights: bf16 ~3.1 GB | 8-bit ~1.7 GB | 4-bit ~0.9 GB  (LoRA 
params, optimizer state and activations come on top)
   bf16: fits
  8-bit: fits
  4-bit: fits
──────────────────────────────── tunekit train ─────────────────────────────────
tunekit 0.1.0  hardware: cuda: Tesla T4, 15.6 GB (bf16=yes)
data: 98 train / 2 eval  (source format: messages -> messages, dropped 0)
weights: 3.1 GB on disk, ~0.9 GB resident in 4-bit vs 16 GB VRAM: fits (before 
activations)

model.safetensors: downloading bytes:   0% 0.00/3.09G [00:00<?, ?B/s]
model.safetensors: downloading bytes:   2% 54.0M/3.09G [00:01<00:37, 79.9MB/s, 4.04MB/s  ]
model.safetensors: reconstructing file:   1% 28.1M/3.09G [00:01<02:11, 23.2MB/s, 4.26kB/s  ]
model.safetensors: downloading bytes:   4% 134M/3.09G [00:02<00:30, 97.1MB/s, 10.9MB/s  ]
model.safetensors: downloadin

## 4. Did it help?

`tunekit eval` scores the held-out split with the tuned *and* the base model, and prints sample
generations side by side. Lower loss on the assistant turns plus better-looking samples is the signal.

In [6]:
# --load-in-4bit reloads the base quantised, the way it was trained.
!tunekit eval outputs/qlora --samples 3 --max-new-tokens 80

scoring 2 examples from the eval split  (messages)
Loading weights: 100% 338/338 [00:00<00:00, 4283.67it/s]
Loading weights: 100% 338/338 [00:00<00:00, 4356.46it/s]
      held-out loss on 2 examples (37 scored tokens)      
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━┓
┃ model                            ┃   loss ┃ perplexity ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━┩
│ base: Qwen/Qwen2.5-1.5B-Instruct │ 4.9060 │    135.095 │
│ tuned: outputs/qlora             │ 2.2814 │      9.791 │
└──────────────────────────────────┴────────┴────────────┘
tuned loss is lower than base by 2.6246 nats/token
─────────────────────────────────── sample 0 ───────────────────────────────────
prompt: What's 2 + 2? Explain like I'm five.
reference: Two an' two be four, as sure as the tide, matey!
base: Sure! Let's think about it together.

Imagine you have two apples and your friend gives you another apple. Now, if we 
count all the apples, how many would that be?

1. First appl

## 5. Chat with it from Python

In [7]:
from tunekit.inference import load_for_inference, stream_reply

# load_in_4bit matches how it was trained; without it the base loads in bf16,
# which for a 7B adapter would not fit a T4.
model, tok = load_for_inference("outputs/qlora", load_in_4bit=True)

def ask(q, **kw):
    print("".join(stream_reply(model, tok, [{"role": "user", "content": q}], **kw)))

ask("What is the capital of France?", max_new_tokens=120)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

The French capital is Paris!


## 6. Export (optional)

Merge the adapter into the base model, then convert to GGUF for Ollama / LM Studio / llama.cpp.

Note we install `tunekit[gguf]` rather than llama.cpp's own `requirements-convert_hf_to_gguf.txt`:
those pins would downgrade transformers and torch and break the environment above.

In [8]:
!tunekit merge outputs/qlora outputs/merged --dtype float16
!git clone -q --depth 1 https://github.com/ggml-org/llama.cpp
!pip install -q "tunekit[gguf]"
!tunekit export outputs/merged --quant q8_0 --llama-cpp llama.cpp --ollama
!ls -lh outputs/merged/*.gguf outputs/merged/Modelfile

merging outputs/qlora into Qwen/Qwen2.5-1.5B-Instruct (float16)
Loading weights: 100% 338/338 [00:02<00:00, 125.04it/s]
Writing model shards: 100% 1/1 [02:24<00:00, 144.75s/it]
merged model saved to outputs/merged
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 kB 10.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
wandb 0.28.1 requires protobuf!=5.28.0,!=5.29.0,<8,>=5, but you have protobuf 4.25.9 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 4.25.9 which is incompatible.
grain 0.2.18 requires protobuf>=5.28.3, but you have protobuf 4.25.9 which is incompatible.
tensorflow 2.20.0 requires protobuf>=5.28.0, but you have protobuf 4.25.9 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 4.25.9 which is incompatible.
$ /usr/bin/python3 llama.cpp

In [9]:
#@title Download the adapter (small — a few MB, unlike the merged model)
!zip -qr adapter.zip outputs/qlora -x "outputs/qlora/checkpoint-*"
from google.colab import files
files.download("adapter.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 7. Report

Prints everything needed for a hardware report: the GPU, peak VRAM, losses and throughput.
Paste this output into an issue at github.com/Dolmaa24/tunekit/issues — it is what keeps the
VRAM table in the README honest.

In [10]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!tunekit info outputs/qlora

Tesla T4, 15360 MiB
tunekit 0.1.0
hardware: cuda: Tesla T4, 15.6 GB (bf16=yes)
bitsandbytes (QLoRA): available
{
  "tunekit_version": "0.1.0",
  "base_model": "Qwen/Qwen2.5-1.5B-Instruct",
  "lora": true,
  "kind": "messages",
  "train_examples": 98,
  "eval_examples": 2,
  "hardware": {
    "device": "cuda",
    "name": "Tesla T4",
    "bf16": true,
    "fp16": true,
    "vram_gb": 15.637086208,
    "n_gpus": 1
  },
  "peak_vram_gb": 3.04,
  "metrics": {
    "train_runtime": 79.6185,
    "train_samples_per_second": 2.462,
    "train_steps_per_second": 0.176,
    "total_flos": 104667341856768.0,
    "train_loss": 2.0417377267565047,
    "epoch": 2.0,
    "eval_loss": 1.0438669919967651,
    "eval_runtime": 0.3072,
    "eval_samples_per_second": 6.511,
    "eval_steps_per_second": 3.256,
    "eval_entropy": 1.2311687469482422,
    "eval_num_tokens": 11906.0,
    "eval_mean_token_accuracy": 0.782608687877655
  },
  "wall_time_s": 139.0
}
